# Анализ центральности графа дружбы VK

**Задача:** для студентов учебной и параллельной групп построить граф дружеских связей в VK и определить:
- центральность по степени (количество связей);
- центральность по посредничеству (betweenness centrality);
- центральность по близости (closeness centrality);
- студента с наибольшим количеством связей.

### Модель графа

- **Вершина** — студент из исходного файла.
- **Ребро** — два студента являются друзьями в VK.

## 1. Установка библиотек

In [ ]:
import os
import re
import time
from getpass import getpass
from itertools import combinations
from collections import Counter
import random
import colorsys
import math
import numpy as np

import pandas as pd
import networkx as nx
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 2. Настройки

Подготовка файла "data_VK.xlsx" к загрузке

In [4]:
INPUT_FILE = "data_VK.xlsx"

# Имя листа Excel. Для первого листа можно оставить 0
EXCEL_SHEET = 0

VK_API_VERSION = "5.199"

REQUEST_DELAY = 1.0

MAX_RETRIES = 5

EXECUTE_BATCH = 25  # узлов за один execute-запрос


## 3. Загрузка списка студентов

Ячейка пытается самостоятельно найти подходящий столбец с ID.

In [7]:
def load_students(path):
    path = str(path)
    ext = os.path.splitext(path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path, sheet_name=EXCEL_SHEET)
    elif ext == ".csv":
        try:
            df = pd.read_csv(path)
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="cp1251")
    else:
        raise ValueError("Поддерживаются только XLSX, XLS и CSV.")

    df.columns = [str(c).strip().lower() for c in df.columns]

    id_candidates = ["id", "vk_id", "user_id", "userid", "вк id", "vk"]
    id_col = next((c for c in id_candidates if c in df.columns), None)

    if id_col is None:
        best_col = None
        best_score = 0
        for col in df.columns:
            values = df[col].dropna().astype(str)
            if len(values) == 0:
                continue
            score = values.str.contains(
                r"(?:vk\.com/)?(?:id)?\d+|^\d+$",
                case=False,
                regex=True
            ).mean()
            if score > best_score:
                best_score = score
                best_col = col

        if best_col is None or best_score < 0.5:
            raise ValueError(
                "Не удалось определить столбец с VK ID. "
                "Переименуйте его в 'id'."
            )
        id_col = best_col

    result = pd.DataFrame()
    result["id_raw"] = df[id_col]

    if "name" in df.columns:
        result["name"] = df["name"].fillna("").astype(str).str.strip()
    elif "фио" in df.columns:
        result["name"] = df["фио"].fillna("").astype(str).str.strip()
    else:
        result["name"] = ""

    if "group" in df.columns:
        result["group"] = df["group"].fillna("Не указана").astype(str).str.strip()
    elif "группа" in df.columns:
        result["group"] = df["группа"].fillna("Не указана").astype(str).str.strip()
    else:
        result["group"] = "Не указана"

    def parse_vk_id(value):
        s = str(value).strip()
        # URL вида https://vk.com/id123 или vk.com/id123
        m = re.search(r"(?:vk\.com/)(?:id)?(\d+)", s, re.I)
        if m:
            return int(m.group(1))
        # Просто число
        m = re.fullmatch(r"(\d+)", s)
        if m:
            return int(m.group(1))
        # id123
        m = re.fullmatch(r"id(\d+)", s, re.I)
        if m:
            return int(m.group(1))
        # VK юзернейм: буквы, цифры, точки, подчёркивания, дефисы (мин. 2 символа)
        m = re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9_.\-]+", s)
        if m:
            return s  # строка — будет разрешена через users.get
        return pd.NA

    result["id"] = result["id_raw"].map(parse_vk_id)
    result = result.dropna(subset=["id"]).copy()
    # Не кастуем в int64 сейчас — юзернеймы будут разрешены через API

    result = result.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return result

students = load_students(INPUT_FILE)

print(f"Количество студентов в выборке: {len(students)}")
display(students.head(10))

Количество студентов в выборке: 28


,id_raw,name,group,id
0,metonimiaa,,BSMO-11-26,metonimiaa
1,bignigga,,BSMO-12-26,bignigga
2,kotletosss,,BSMO-11-26,kotletosss
3,y.doyneko,,BSMO-12-26,y.doyneko
4,id422397933,,BSMO-11-26,422397933
5,saulinivan,,BSMO-12-26,saulinivan
6,antonlevn,,BSMO-11-26,antonlevn
7,dimitr101,,BSMO-12-26,dimitr101
8,s.abramov2,,BSMO-11-26,s.abramov2
9,ku_ku_riz,,BSMO-12-26,ku_ku_riz


## 4. Проверка выборки

In [8]:
print("Количество студентов по группам:")
display(students["group"].value_counts().rename_axis("group").to_frame("count"))

print("\nДубликаты ID:", students["id"].duplicated().sum())
print("Пропуски ID:", students["id"].isna().sum())

Количество студентов по группам:


,count
group,
BSMO-11-26,17
BSMO-12-26,11



Дубликаты ID: 0
Пропуски ID: 0


## 5. Токен VK API

Код ниже хранит токен только в памяти текущей сессии Jupyter.

In [9]:
VK_TOKEN = getpass("Введите VK access token: ").strip()

if not VK_TOKEN:
    raise ValueError("Токен не введён.")

## 6. Вызов VK API

Используются методы `users.get` для профилей и `friends.get` для списка друзей.

In [10]:
API_URL = "https://api.vk.com/method/"

session = requests.Session()

def vk_request(method, **params):
    params["access_token"] = VK_TOKEN
    params["v"] = VK_API_VERSION

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                API_URL + method,
                params=params,
                timeout=20
            )
            response.raise_for_status()
            data = response.json()

            if "error" in data:
                error = data["error"]
                code = error.get("error_code")
                msg = error.get("error_msg", "Неизвестная ошибка VK API")
                if code == 9:
                    wait = 5 * attempt
                    print(f"  [flood control] ждём {wait}s...")
                    time.sleep(wait)
                    last_error = RuntimeError(f"VK API error {code}: {msg}")
                    continue
                raise RuntimeError(f"VK API error {code}: {msg}")

            return data["response"]

        except RuntimeError:
            raise
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 1.5)

    raise RuntimeError(
        f"Не удалось выполнить запрос {method}: {last_error}"
    ) from last_error


## 7. Получение профилей студентов

Сохраняем только минимально необходимые сведения: ID, имя и фамилию. Дополнительные персональные поля не собираются.

In [11]:
student_identifiers = students["id"].tolist()  # числа и/или юзернеймы

profiles = []
identifier_order = []

for start in range(0, len(student_identifiers), 1000):
    batch = student_identifiers[start:start + 1000]
    resp = vk_request(
        "users.get",
        user_ids=",".join(map(str, batch))
    )
    profiles.extend(resp)
    identifier_order.extend(batch)
    time.sleep(REQUEST_DELAY)

profiles_df = pd.DataFrame(profiles)

if not profiles_df.empty:
    # Маппинг: оригинальный идентификатор (юзернейм или число) -> числовой id VK
    id_resolution = {orig: prof["id"] for orig, prof in zip(identifier_order, profiles)}

    students["id"] = students["id"].map(lambda x: id_resolution.get(x, x))
    students["id"] = pd.to_numeric(students["id"], errors="coerce")
    students = students.dropna(subset=["id"]).copy()
    students["id"] = students["id"].astype("int64")

    profiles_df["name_vk"] = (
        profiles_df["first_name"].fillna("") + " " +
        profiles_df["last_name"].fillna("")
    ).str.strip()

    if "deactivated" not in profiles_df.columns:
        profiles_df["deactivated"] = None

    students = students.drop(columns=["name"], errors="ignore").merge(
        profiles_df[["id", "name_vk", "deactivated"]],
        on="id",
        how="left"
    )

    students["name"] = students["name_vk"].fillna("")
    students = students.drop(columns=["name_vk"], errors="ignore")

else:
    students["name"] = students.get("name", "")
    students["deactivated"] = None

print(f"Загружено студентов: {len(students)}")
display(students.head())

Загружено студентов: 28


,id_raw,group,id,deactivated,name
0,metonimiaa,BSMO-11-26,214576903,None,Артем Кузьмин
1,bignigga,BSMO-12-26,447383697,None,Игорь Сусаров
2,kotletosss,BSMO-11-26,147436715,None,Мария Александрова
3,y.doyneko,BSMO-12-26,365680993,None,Юлия Дойнеко
4,id422397933,BSMO-11-26,422397933,None,Андрей Клейменов


## 8. Получение списков друзей

Для каждого студента запрашивается список его друзей. API может не вернуть список полностью или может отказать в доступе для конкретного профиля. Такие случаи фиксируются отдельно и не скрываются.

In [12]:
def get_all_friends(user_id, count=5000):
    all_friends = []
    offset = 0

    while True:
        response = vk_request(
            "friends.get",
            user_id=user_id,
            count=count,
            offset=offset
        )

        items = response.get("items", [])
        total = response.get("count", 0)

        all_friends.extend(items)

        if not items or len(all_friends) >= total or len(items) < count:
            break

        offset += len(items)
        time.sleep(REQUEST_DELAY)

    return set(all_friends)


friends_by_user = {}
api_errors = []

for i, user_id in enumerate(students["id"], start=1):
    try:
        friends_by_user[int(user_id)] = get_all_friends(int(user_id))
        print(f"[{i}/{len(students)}] ID {user_id}: "
              f"{len(friends_by_user[int(user_id)])} друзей")
    except Exception as exc:
        api_errors.append({
            "user_id": int(user_id),
            "error": str(exc)
        })
        friends_by_user[int(user_id)] = set()
        print(f"[{i}/{len(students)}] ID {user_id}: ОШИБКА — {exc}")

    time.sleep(REQUEST_DELAY)

errors_df = pd.DataFrame(api_errors)
print(f"\nОшибок API: {len(errors_df)}")
if not errors_df.empty:
    display(errors_df)

def get_friends_batch_execute(user_ids: list, count: int = 5000) -> dict:
    """Получает друзей для батча через VK execute — 25 calls в 1 запросе."""
    code_lines = [
        f'API.friends.get({{"user_id":{uid},"count":{count}}})'
        for uid in user_ids
    ]
    code = "return [" + ",".join(code_lines) + "];"
    try:
        results = vk_request("execute", code=code)
    except Exception:
        return {uid: set() for uid in user_ids}
    friends_map = {}
    for uid, result in zip(user_ids, results or []):
        if result and isinstance(result, dict):
            friends_map[uid] = set(result.get("items", []))
        else:
            friends_map[uid] = set()
    return friends_map


[1/28] ID 214576903: 126 друзей
[2/28] ID 447383697: 104 друзей
[3/28] ID 147436715: 112 друзей
[4/28] ID 365680993: 289 друзей
[5/28] ID 422397933: 5413 друзей
[6/28] ID 223966628: 153 друзей
[7/28] ID 708207289: 111 друзей
[8/28] ID 299547783: 272 друзей
[9/28] ID 354792455: 88 друзей
[10/28] ID 288851914: 105 друзей
[11/28] ID 437474057: 40 друзей
[12/28] ID 465105120: 181 друзей
[13/28] ID 193511426: ОШИБКА — Не удалось выполнить запрос friends.get: VK API error 30: This profile is private
[14/28] ID 198182178: ОШИБКА — Не удалось выполнить запрос friends.get: VK API error 30: This profile is private
[15/28] ID 229017385: 20 друзей
[16/28] ID 473310225: 177 друзей
[17/28] ID 133821410: 127 друзей
[18/28] ID 424935969: 30 друзей
[19/28] ID 172152768: 62 друзей
[20/28] ID 139159408: 275 друзей
[21/28] ID 677045683: 108 друзей
[22/28] ID 236434667: ОШИБКА — Не удалось выполнить запрос friends.get: VK API error 30: This profile is private
[23/28] ID 278508214: 104 друзей
[24/28] ID 392

,user_id,error
0,193511426,Не удалось выполнить запрос friends.get: VK AP...
1,198182178,Не удалось выполнить запрос friends.get: VK AP...
2,236434667,Не удалось выполнить запрос friends.get: VK AP...


## 9. Построение графа

Ребро добавляется только между двумя студентами из исходной выборки, если первый студент видит второго в своём списке друзей VK.

Для дружбы VK граф рассматривается как **неориентированный**: наличие связи `A — B` означает одно ребро независимо от того, из чьего списка друзей оно было обнаружено.

In [13]:
student_id_set = set(students["id"].astype(int))

G = nx.Graph()

# Добавляем все вершины заранее, включая студентов без найденных связей
for _, row in students.iterrows():
    G.add_node(
        int(row["id"]),
        name=row["name"] if row["name"] else str(row["id"]),
        group=row["group"]
    )

# Проверяем только связи внутри выборки.
for user_id, friend_ids in friends_by_user.items():
    internal_friends = student_id_set.intersection(friend_ids)

    for friend_id in internal_friends:
        if int(user_id) != int(friend_id):
            G.add_edge(int(user_id), int(friend_id))

print("Вершин:", G.number_of_nodes())
print("Рёбер:", G.number_of_edges())
print("Компонент связности:", nx.number_connected_components(G))

Вершин: 28
Рёбер: 16
Компонент связности: 16


In [ ]:
print("=" * 45)
print("СВОДКА ГРАФА (исходный, студенты)")
print("=" * 45)
print(f"  Вершин (студентов):       {G.number_of_nodes()}")
print(f"  Рёбер (дружеских связей): {G.number_of_edges()}")
print(f"  Компонент связности:      {nx.number_connected_components(G)}")
print(f"  Изолированных вершин:     {sum(1 for n in G.nodes() if G.degree(n) == 0)}")
density = nx.density(G)
print(f"  Плотность графа:          {density:.4f}")
print("=" * 45)

## 10. Таблица рёбер

Эта таблица показывает, кто с кем является другом внутри учебной выборки.

In [14]:
id_to_name = dict(zip(students["id"], students["name"]))
id_to_group = dict(zip(students["id"], students["group"]))

edges = []
for u, v in G.edges():
    edges.append({
        "id_1": u,
        "name_1": id_to_name.get(u, str(u)),
        "group_1": id_to_group.get(u, ""),
        "id_2": v,
        "name_2": id_to_name.get(v, str(v)),
        "group_2": id_to_group.get(v, "")
    })

edges_df = pd.DataFrame(edges)
display(edges_df)

,id_1,name_1,group_1,id_2,name_2,group_2
0,365680993,Юлия Дойнеко,BSMO-12-26,223966628,Иван Саулин,BSMO-12-26
1,365680993,Юлия Дойнеко,BSMO-12-26,299547783,Дмитрий Полуэктов,BSMO-12-26
2,422397933,Андрей Клейменов,BSMO-11-26,193511426,Александр Васильев,BSMO-11-26
3,422397933,Андрей Клейменов,BSMO-11-26,299547783,Дмитрий Полуэктов,BSMO-12-26
4,422397933,Андрей Клейменов,BSMO-11-26,437474057,Дмитрий Базлов,BSMO-11-26
5,422397933,Андрей Клейменов,BSMO-11-26,229017385,Кирилл Кучеренко,BSMO-11-26
6,422397933,Андрей Клейменов,BSMO-11-26,236434667,Александр Голоктионов,BSMO-12-26
7,223966628,Иван Саулин,BSMO-12-26,299547783,Дмитрий Полуэктов,BSMO-12-26
8,354792455,Сергей Абрамов,BSMO-11-26,278508214,Илья Руднев,BSMO-11-26
9,354792455,Сергей Абрамов,BSMO-11-26,241970575,Никита Марков,BSMO-11-26


## 11. Расчёт центральностей

Используются четыре показателя:

**Центральность по степени** — доля вершин, с которыми человек непосредственно связан. Для поиска человека с максимальным количеством связей дополнительно рассматривается обычная степень `degree`.

**Центральность по посредничеству** (`betweenness`) — показывает, насколько часто вершина оказывается на кратчайших путях между другими вершинами. Высокое значение характерно для человека, который может выступать связующим звеном между частями группы.

**Центральность по близости** (`closeness`) — показывает, насколько быстро вершина достижима от остальных связанных с ней вершин по кратчайшим путям.

**Центральность собственного вектора** (`eigenvector`) — учитывает не только количество связей, но и их «качество»: связь с высокоцентральным человеком даёт больше веса. Если граф несвязный, вычисляется внутри каждой компоненты.

Если граф распадается на несколько компонент, значения closeness и eigenvector нужно интерпретировать осторожно: вершины из разных компонент недостижимы друг из друга.

In [ ]:
degree = dict(G.degree())
degree_centrality = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G, normalized=True)
closeness = nx.closeness_centrality(G)

# Eigenvector centrality на несвязном графе вычисляется покомпонентно.
# eigenvector_centrality_numpy падает с TypeError на изолированных вершинах.
eigenvector = {}
for component in nx.connected_components(G):
    sub = G.subgraph(component).copy()
    if sub.number_of_edges() == 0:
        for n in sub.nodes():
            eigenvector[n] = 0.0
    else:
        try:
            sub_ec = nx.eigenvector_centrality_numpy(sub)
            eigenvector.update(sub_ec)
        except Exception:
            for n in sub.nodes():
                eigenvector[n] = 0.0

results = students[["id", "name", "group"]].copy()
results["degree"] = results["id"].map(degree).fillna(0).astype(int)
results["degree_centrality"] = results["id"].map(degree_centrality).fillna(0)
results["betweenness_centrality"] = results["id"].map(betweenness).fillna(0)
results["closeness_centrality"] = results["id"].map(closeness).fillna(0)
results["eigenvector_centrality"] = results["id"].map(eigenvector).fillna(0)

# --- Таблица 1: Центральность по степени ---
degree_table = (
    results[["id", "name", "group", "degree", "degree_centrality"]]
    .sort_values("degree_centrality", ascending=False)
    .reset_index(drop=True)
)
print("Таблица 1 — Центральность по степени (Degree Centrality)")
display(degree_table.style.format({"degree_centrality": "{:.4f}"}))

# --- Таблица 2: Центральность по посредничеству ---
betweenness_table = (
    results[["id", "name", "group", "betweenness_centrality"]]
    .sort_values("betweenness_centrality", ascending=False)
    .reset_index(drop=True)
)
print("\nТаблица 2 — Центральность по посредничеству (Betweenness Centrality)")
display(betweenness_table.style.format({"betweenness_centrality": "{:.4f}"}))

# --- Таблица 3: Центральность по близости ---
closeness_table = (
    results[["id", "name", "group", "closeness_centrality"]]
    .sort_values("closeness_centrality", ascending=False)
    .reset_index(drop=True)
)
print("\nТаблица 3 — Центральность по близости (Closeness Centrality)")
display(closeness_table.style.format({"closeness_centrality": "{:.4f}"}))

# --- Таблица 4: Центральность собственного вектора ---
eigenvector_table = (
    results[["id", "name", "group", "eigenvector_centrality"]]
    .sort_values("eigenvector_centrality", ascending=False)
    .reset_index(drop=True)
)
print("\nТаблица 4 — Центральность собственного вектора (Eigenvector Centrality)")
display(eigenvector_table.style.format({"eigenvector_centrality": "{:.4f}"}))

# Итоговая сортировка results для ячеек ниже (разделы 12, 16)
results = results.sort_values(
    ["degree", "betweenness_centrality", "closeness_centrality"],
    ascending=False
).reset_index(drop=True)

## 12. Студент с наибольшим количеством связей

In [16]:
max_degree = results["degree"].max()
leaders = results[results["degree"] == max_degree].copy()

print(f"Максимальное количество связей: {max_degree}")
display(
    leaders[[
        "id", "name", "group", "degree",
        "degree_centrality",
        "betweenness_centrality",
        "closeness_centrality",
        "eigenvector_centrality",
    ]]
)

Максимальное количество связей: 5


,id,name,group,degree,degree_centrality,betweenness_centrality,closeness_centrality,eigenvector_centrality
0,422397933,Андрей Клейменов,BSMO-11-26,5,0.185185,0.048433,0.201646,0.593061


## 14. Визуализация графа

Размер вершины пропорционален числу связей. Подписываются имена студентов.

In [ ]:
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.family"] = "DejaVu Sans"

if G.number_of_nodes() == 0:
    print("Граф пуст: нет вершин для отображения.")

else:
    nodes = list(G.nodes())
    n = len(nodes)

    fig, ax = plt.subplots(figsize=(23, 17))

    # Уникальные цвета вершин
    random.seed(42)
    hues = [i / n for i in range(n)]
    random.shuffle(hues)

    node_colors = {}
    for node, hue in zip(nodes, hues):
        saturation = random.uniform(0.65, 0.85)
        lightness = random.uniform(0.45, 0.62)
        rgb = colorsys.hls_to_rgb(hue, lightness, saturation)
        node_colors[node] = "#{:02x}{:02x}{:02x}".format(
            int(rgb[0] * 255),
            int(rgb[1] * 255),
            int(rgb[2] * 255)
        )

    # Расположение вершин
    pos = nx.spring_layout(
        G,
        seed=42,
        k=2.5 / math.sqrt(n),
        iterations=300,
        weight=None
    )

    degrees = dict(G.degree())
    node_sizes = [450 + 220 * degrees[node] for node in nodes]

    # Рёбра графа
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edge_color="#9AA4B2",
        width=1.1,
        alpha=0.32
    )

    # Вершины
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        nodelist=nodes,
        node_color=[node_colors[node] for node in nodes],
        node_size=node_sizes,
        edgecolors="white",
        linewidths=1.8,
        alpha=0.98
    )

    # Подписи рядом с маркерами и пунктирные линии-указатели
    rng = np.random.default_rng(42)

    for node in nodes:
        x, y = pos[node]
        name = str(G.nodes[node].get("name", node))

        # Направление подписи от центра графа наружу
        direction = np.array([x, y], dtype=float)
        norm = np.linalg.norm(direction)

        if norm > 0:
            direction /= norm
        else:
            direction = rng.normal(size=2)
            direction /= np.linalg.norm(direction)

        # Небольшое дополнительное смещение для уменьшения наложений
        direction += rng.normal(0, 0.22, size=2)
        direction /= np.linalg.norm(direction)

        offset = 0.075
        label_x = x + direction[0] * offset
        label_y = y + direction[1] * offset

        # Горизонтальное выравнивание в зависимости от направления
        ha = "left" if direction[0] >= 0 else "right"
        va = "bottom" if direction[1] >= 0 else "top"

        # Пунктирный указатель
        ax.annotate(
            "",
            xy=(x, y),
            xytext=(label_x, label_y),
            arrowprops=dict(
                arrowstyle="-",
                linestyle=(0, (2, 2)),
                color=node_colors[node],
                lw=1.2,
                alpha=0.9
            ),
            zorder=3
        )

        # Текст с белым фоном
        ax.text(
            label_x,
            label_y,
            name,
            fontsize=8.5,
            color="#202938",
            ha=ha,
            va=va,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                edgecolor=node_colors[node],
                linewidth=0.9,
                alpha=0.96
            ),
            zorder=4
        )

    # Нижняя легенда: имя и количество связей
    legend_handles = []

    for node in nodes:
        name = str(G.nodes[node].get("name", node))
        legend_handles.append(
            plt.Line2D(
                [0], [0],
                marker="o",
                linestyle="None",
                markerfacecolor=node_colors[node],
                markeredgecolor="#555555",
                markersize=7,
                label=f"{name} — связей: {degrees[node]}"
            )
        )

    legend_columns = min(4, max(1, math.ceil(n / 12)))

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.012),
        ncol=legend_columns,
        fontsize=8.5,
        frameon=True,
        fancybox=True,
        title="Студенты и количество связей",
        title_fontsize=10,
        columnspacing=1.4,
        handletextpad=0.5,
        labelspacing=0.7
    )

    ax.set_title(
        f"Граф дружеских связей VK внутри двух учебных групп\n"
        f"Вершин: {G.number_of_nodes()}  |  Рёбер: {G.number_of_edges()}",
        fontsize=19,
        fontweight="bold",
        pad=22
    )

    ax.axis("off")

    # Оставляем место для подписей и легенды
    fig.subplots_adjust(
        left=0.06,
        right=0.94,
        top=0.92,
        bottom=min(0.42, 0.12 + 0.025 * math.ceil(n / legend_columns))
    )

    plt.show()


## 15. Дополнительный анализ компонент

Посмотрим, распадается ли группа на отдельные сообщества.

In [19]:
components = sorted(nx.connected_components(G), key=len, reverse=True)

component_rows = []
for idx, component in enumerate(components, start=1):
    for node in component:
        component_rows.append({
            "component": idx,
            "component_size": len(component),
            "id": node,
            "name": id_to_name.get(node, str(node)),
            "group": id_to_group.get(node, "")
        })

components_df = pd.DataFrame(component_rows)
display(components_df)

,component,component_size,id,name,group
0,1,8,365680993,Юлия Дойнеко,BSMO-12-26
1,1,8,193511426,Александр Васильев,BSMO-11-26
2,1,8,223966628,Иван Саулин,BSMO-12-26
3,1,8,299547783,Дмитрий Полуэктов,BSMO-12-26
4,1,8,437474057,Дмитрий Базлов,BSMO-11-26
5,1,8,229017385,Кирилл Кучеренко,BSMO-11-26
6,1,8,236434667,Александр Голоктионов,BSMO-12-26
7,1,8,422397933,Андрей Клейменов,BSMO-11-26
8,2,3,241970575,Никита Марков,BSMO-11-26
9,2,3,278508214,Илья Руднев,BSMO-11-26


## 16. Сохранение результатов

In [20]:
os.makedirs("vk_graph_results", exist_ok=True)

results.to_csv(
    "vk_graph_results/centrality_results.csv",
    index=False,
    encoding="utf-8-sig"
)

edges_df.to_csv(
    "vk_graph_results/edges.csv",
    index=False,
    encoding="utf-8-sig"
)

components_df.to_csv(
    "vk_graph_results/components.csv",
    index=False,
    encoding="utf-8-sig"
)

if not errors_df.empty:
    errors_df.to_csv(
        "vk_graph_results/api_errors.csv",
        index=False,
        encoding="utf-8-sig"
    )

with pd.ExcelWriter(
    "vk_graph_results/vk_graph_analysis.xlsx",
    engine="openpyxl"
) as writer:
    results.to_excel(writer, sheet_name="Centrality_All", index=False)
    degree_table.to_excel(writer, sheet_name="Degree_Centrality", index=False)
    betweenness_table.to_excel(writer, sheet_name="Betweenness_Centrality", index=False)
    closeness_table.to_excel(writer, sheet_name="Closeness_Centrality", index=False)
    eigenvector_table.to_excel(writer, sheet_name="Eigenvector_Centrality", index=False)
    edges_df.to_excel(writer, sheet_name="Edges", index=False)
    components_df.to_excel(writer, sheet_name="Components", index=False)
    if not errors_df.empty:
        errors_df.to_excel(writer, sheet_name="API_errors", index=False)

print("Результаты сохранены в папке vk_graph_results/")
print("Листы Excel:")
print("  Centrality_All          — все метрики")
print("  Degree_Centrality       — центральность по степени")
print("  Betweenness_Centrality  — центральность по посредничеству")
print("  Closeness_Centrality    — центральность по близости")
print("  Eigenvector_Centrality  — центральность собственного вектора")
print("  Edges                   — таблица рёбер")
print("  Components              — компоненты связности")

Результаты сохранены в папке vk_graph_results/


In [ ]:
# --- Шаг 1: найти общих друзей (присутствуют в списке 2+ студентов) ---

MIN_SHARED = 2  # минимум студентов, которые знают этого человека

try:
    student_id_set_int = student_id_set
except NameError:
    student_id_set_int = set(
        pd.to_numeric(students["id"], errors="coerce").dropna().astype("int64")
    )

# fid -> список student_id, у которых fid есть в друзьях
friend_seen_by: dict[int, list[int]] = {}
for sid, fset in friends_by_user.items():
    for fid in fset:
        if fid not in student_id_set_int:
            friend_seen_by.setdefault(fid, []).append(sid)

shared_friends = {
    fid: sids
    for fid, sids in friend_seen_by.items()
    if len(sids) >= MIN_SHARED
}

print(f"Всего уникальных друзей студентов: {len(friend_seen_by)}")
print(f"Общих друзей (знакомы с {MIN_SHARED}+ студентами): {len(shared_friends)}")

# Распределение: сколько людей знакомы с N студентами
dist = Counter(len(v) for v in shared_friends.values())
for k in sorted(dist):
    print(f"  знают {k} студентов: {dist[k]} человек")

In [22]:
# --- Шаг 2: загрузить профили общих друзей ---

shared_ids = list(shared_friends.keys())

new_profiles_raw = []
for i in range(0, len(shared_ids), 1000):
    batch = shared_ids[i:i + 1000]
    resp = vk_request("users.get", user_ids=",".join(map(str, batch)))
    new_profiles_raw.extend(resp)
    time.sleep(REQUEST_DELAY)

new_profiles_df = pd.DataFrame(new_profiles_raw)
new_profiles_df["name_vk"] = (
    new_profiles_df["first_name"].fillna("") + " " +
    new_profiles_df["last_name"].fillna("")
).str.strip()

if "deactivated" not in new_profiles_df.columns:
    new_profiles_df["deactivated"] = None

# Убираем удалённые/заблокированные аккаунты
new_profiles_df = new_profiles_df[new_profiles_df["deactivated"].isna()].copy()

print(f"Загружено профилей общих друзей: {len(new_profiles_df)}")
display(new_profiles_df[["id", "name_vk"]].head(10))

Загружено профилей общих друзей: 228


,id,name_vk
0,444827717,Артемий Паршин
1,223364702,Макс Гончаров
2,359450337,Макс Глебов
3,467160317,Лиса Косарева
4,96595205,Ярослав Акатьев
5,254781729,Яна Алексеева
6,384742759,Игорь Соболев
7,240200561,Ангелина Батракова
8,387169661,Олег Погребняк
9,494481372,Егор Михайлов


In [23]:
# --- Шаг 3: запросить списки друзей у общих друзей (батчами через execute) ---
active_shared_ids = set(new_profiles_df["id"].tolist())

query_list = sorted(
    [fid for fid in shared_friends if fid in active_shared_ids],
    key=lambda fid: len(shared_friends[fid]),
    reverse=True
)

print(f"Будем запрашивать друзей у {len(query_list)} общих друзей...")

friends_of_shared: dict = {}
fof_errors = []

for i in range(0, len(query_list), EXECUTE_BATCH):
    batch = query_list[i:i + EXECUTE_BATCH]
    batch_results = get_friends_batch_execute(batch)
    for fid, fset in batch_results.items():
        if fset:
            friends_of_shared[fid] = fset
        else:
            friends_of_shared[fid] = set()
            fof_errors.append({"user_id": fid, "error": "empty or private"})
    done = min(i + EXECUTE_BATCH, len(query_list))
    if done % 50 == 0 or done == len(query_list):
        print(f"  [{done}/{len(query_list)}] обработано")
    time.sleep(REQUEST_DELAY)

print(f"\nОшибок: {len(fof_errors)}")


Будем запрашивать друзей у 228 общих друзей...
  [20/228] обработано
  [40/228] обработано
  [60/228] обработано
  [80/228] обработано
  [100/228] обработано
  [120/228] обработано
  [140/228] обработано
  [160/228] обработано
  [180/228] обработано
  [200/228] обработано
  [220/228] обработано
  [228/228] обработано

Ошибок: 39


In [ ]:
# --- Шаг 4: построить расширенный граф (BFS до 300 000 рёбер) ---
TARGET_EDGES = 300_000
MAX_NODES    = 100_000
BATCH_SIZE   = 100

G_ext = G.copy()
id_to_name_ext = dict(id_to_name)
id_to_group_ext = dict(id_to_group)
name_map = {row["id"]: row["name_vk"] for _, row in new_profiles_df.iterrows()}

# Слой 3
for fid, sids in shared_friends.items():
    if fid not in active_shared_ids:
        continue
    name = name_map.get(fid, str(fid))
    id_to_name_ext[fid] = name
    id_to_group_ext[fid] = "Друг друга"
    G_ext.add_node(fid, name=name, group="Друг друга")
    for sid in sids:
        G_ext.add_edge(fid, int(sid))

# Слой 4
for fid, their_friends in friends_of_shared.items():
    for other_fid in their_friends:
        if other_fid == fid:
            continue
        if other_fid not in G_ext.nodes():
            G_ext.add_node(other_fid, name=str(other_fid), group="Слой 4")
            id_to_name_ext[other_fid] = str(other_fid)
            id_to_group_ext[other_fid] = "Слой 4"
        G_ext.add_edge(fid, other_fid)

print(f"После слоёв 3+4 (без доп. запросов):")
print(f"  Вершин: {G_ext.number_of_nodes():,}  |  Рёбер: {G_ext.number_of_edges():,}")
print()

# --- BFS-расширение через execute (батчи по EXECUTE_BATCH) ---
fetched_set: set = set(friends_by_user.keys()) | set(friends_of_shared.keys())

iteration = 0
print(f"BFS-расширение до {TARGET_EDGES:,} рёбер (execute батчи по {EXECUTE_BATCH})...")
print(f"{'Итер':>5}  {'Вершин':>8}  {'Рёбер':>9}  {'Добавлено рёбер':>16}")
print("─" * 48)

while G_ext.number_of_edges() < TARGET_EDGES:
    candidates = sorted(
        [n for n in G_ext.nodes() if n > 0 and n not in fetched_set],
        key=lambda n: G_ext.degree(n),
        reverse=True
    )[:BATCH_SIZE]

    if not candidates:
        print("Нет новых кандидатов — расширение завершено.")
        break

    edges_before = G_ext.number_of_edges()
    fetched_set.update(candidates)

    for b_start in range(0, len(candidates), EXECUTE_BATCH):
        batch = candidates[b_start:b_start + EXECUTE_BATCH]
        batch_results = get_friends_batch_execute(batch)

        cur_nodes = set(G_ext.nodes())
        for node_id, friends in batch_results.items():
            for fid in friends:
                if fid in cur_nodes:
                    G_ext.add_edge(node_id, fid)
                elif G_ext.number_of_nodes() < MAX_NODES:
                    G_ext.add_node(fid, name=str(fid), group="Слой N")
                    id_to_name_ext[fid] = str(fid)
                    id_to_group_ext[fid] = "Слой N"
                    G_ext.add_edge(node_id, fid)
                    cur_nodes.add(fid)

        time.sleep(REQUEST_DELAY)
        if G_ext.number_of_edges() >= TARGET_EDGES:
            break

    iteration += 1
    added = G_ext.number_of_edges() - edges_before
    print(f"{iteration:>5}  {G_ext.number_of_nodes():>8,}  {G_ext.number_of_edges():>9,}  {added:>+16,}")

print()
print(f"Итог: {G_ext.number_of_nodes():,} вершин, {G_ext.number_of_edges():,} рёбер")


In [ ]:
# --- Шаг 5: сводная таблица ---

ext_rows = []
for node in G_ext.nodes():
    if node in student_id_set_int:
        node_type = "Студент"
        sc = "-"
    elif node in active_shared_ids:
        node_type = "Друг друга"
        sc = len(shared_friends.get(node, []))
    else:
        node_type = id_to_group_ext.get(node, "Слой N")
        sc = 0

    ext_rows.append({
        "id": node,
        "name": id_to_name_ext.get(node, str(node)),
        "group": id_to_group_ext.get(node, ""),
        "type": node_type,
        "degree": G_ext.degree(node),
        "student_connections": sc,
    })

ext_df = (
    pd.DataFrame(ext_rows)
    .sort_values("degree", ascending=False)
    .reset_index(drop=True)
)

print("=" * 48)
print("ИТОГОВЫЙ ГРАФ")
print("=" * 48)
for t in ["Студент", "Друг друга", "Слой 4", "Слой N"]:
    cnt = (ext_df["type"] == t).sum()
    if cnt > 0:
        print(f"  {t:<25} {cnt:>8,}")
print("─" * 48)
print(f"  {'Итого вершин':<25} {len(ext_df):>8,}")
print(f"  {'Итого рёбер':<25} {G_ext.number_of_edges():>8,}")
print("=" * 48)
display(ext_df.head(50))


In [ ]:
plt.rcParams["figure.dpi"] = 130

VIZ_LIMIT = 800   # больше — показываем подграф топ-N + студентов

nodes_all = list(G_ext.nodes())
n_total = len(nodes_all)

if n_total <= VIZ_LIMIT:
    G_viz = G_ext
    viz_note = ""
else:
    top_nodes = sorted(G_ext.nodes(), key=lambda n: G_ext.degree(n), reverse=True)[:VIZ_LIMIT]
    viz_set = set(top_nodes) | student_id_set_int
    G_viz = G_ext.subgraph(viz_set).copy()
    viz_note = f"\n(показан подграф: топ-{len(G_viz)} узлов по степени из {n_total:,})"

nodes_ext = list(G_viz.nodes())
n_ext = len(nodes_ext)

fig, ax = plt.subplots(figsize=(40, 30))

random.seed(7)

node_colors_ext = {}
for node in nodes_ext:
    if node in student_id_set_int:
        hue = random.uniform(0.55, 0.72)
        sat, lig = random.uniform(0.6, 0.85), random.uniform(0.45, 0.62)
    else:
        hue = random.uniform(0.05, 0.18)
        sat, lig = random.uniform(0.7, 0.9), random.uniform(0.50, 0.65)
    rgb = colorsys.hls_to_rgb(hue, lig, sat)
    node_colors_ext[node] = "#{:02x}{:02x}{:02x}".format(
        int(rgb[0] * 255), int(rgb[1] * 255), int(rgb[2] * 255)
    )

pos_ext = nx.spring_layout(
    G_viz,
    seed=7,
    k=30.5 / math.sqrt(n_ext),
    iterations=300,
    weight=None
)

degrees_ext = dict(G_viz.degree())
node_sizes_ext = [350 + 180 * degrees_ext[n] for n in nodes_ext]

nx.draw_networkx_edges(G_viz, pos_ext, ax=ax,
                       edge_color="#9AA4B2", width=0.9, alpha=0.28)

nx.draw_networkx_nodes(G_viz, pos_ext, ax=ax,
                       nodelist=nodes_ext,
                       node_color=[node_colors_ext[n] for n in nodes_ext],
                       node_size=node_sizes_ext,
                       edgecolors="white", linewidths=1.5, alpha=0.97)

# Подписи только для студентов и слоя 3 (чтобы не перегружать при большом графе)
label_nodes = [n for n in nodes_ext if n in student_id_set_int or n in active_shared_ids]
rng2 = np.random.default_rng(7)
for node in label_nodes:
    x, y = pos_ext[node]
    name = str(G_viz.nodes[node].get("name", node))
    direction = np.array([x, y], dtype=float)
    norm = np.linalg.norm(direction)
    direction = direction / norm if norm > 0 else rng2.normal(size=2)
    direction += rng2.normal(0, 0.2, size=2)
    direction /= np.linalg.norm(direction)
    offset = 0.065
    lx, ly = x + direction[0] * offset, y + direction[1] * offset
    ha = "left" if direction[0] >= 0 else "right"
    va = "bottom" if direction[1] >= 0 else "top"
    ax.annotate("", xy=(x, y), xytext=(lx, ly),
                arrowprops=dict(arrowstyle="-", linestyle=(0, (2, 2)),
                                color=node_colors_ext[node], lw=1.1, alpha=0.85), zorder=3)
    ax.text(lx, ly, name, fontsize=7.5, color="#202938", ha=ha, va=va,
            bbox=dict(boxstyle="round,pad=0.22", facecolor="white",
                      edgecolor=node_colors_ext[node], linewidth=0.8, alpha=0.95), zorder=4)

legend_elements = [
    mpatches.Patch(facecolor="#4A90D9", edgecolor="#555", label="Студент"),
    mpatches.Patch(facecolor="#E8943A", edgecolor="#555", label="Друг друга (слой 3)"),
    mpatches.Patch(facecolor="#5CAB4A", edgecolor="#555", label="Слой 4+"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=11,
          frameon=True, fancybox=True)

ax.set_title(
    f"Расширенный граф VK: студенты + друзья друзей{viz_note}\n"
    f"Вершин: {G_ext.number_of_nodes():,}  |  Рёбер: {G_ext.number_of_edges():,}",
    fontsize=20, fontweight="bold", pad=22
)
ax.axis("off")
fig.subplots_adjust(left=0.04, right=0.96, top=0.93, bottom=0.04)
plt.show()


In [ ]:
ext_df.to_csv(
    "vk_graph_results/extended_nodes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Слой 3: общие друзья студентов
friends_layer3 = (
    ext_df[ext_df["type"] == "Друг друга"]
    .copy()
    .sort_values("student_connections", ascending=False)
    .reset_index(drop=True)
)

friends_layer3.to_csv(
    "vk_graph_results/friends_layer3.csv",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    "vk_graph_results/vk_graph_analysis.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer:
    ext_df.to_excel(writer, sheet_name="Extended_Nodes", index=False)
    friends_layer3.to_excel(writer, sheet_name="Friends_Layer3", index=False)

print("Расширенная таблица сохранена:")
print("  vk_graph_results/extended_nodes.csv")
print("  vk_graph_results/friends_layer3.csv")
print("  vk_graph_results/vk_graph_analysis.xlsx  → листы 'Extended_Nodes', 'Friends_Layer3'")
print(f"Итого вершин: {len(ext_df):,}")
print(f"Итого рёбер:  {G_ext.number_of_edges():,}")